In [ ]:
import numpy as np
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, average_precision_score, classification_report
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks
from collections import defaultdict
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [2]:
files = ['two_class_raw_3s_no.csv', 'two_class_raw_3s_yo_0.5.csv'
        'two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/'

In [ ]:
class NestedCVOptimizer:
    def __init__(self, X, y, groups, n_outer_folds=5, n_inner_folds=3, 
                n_trials=100, random_state=42):
        """
        Initialize the nested cross-validation optimizer.
        
        Parameters:
        - X: Feature matrix
        - y: Target labels
        - groups: Group labels for GroupKFold
        - n_outer_folds: Number of outer CV folds
        - n_inner_folds: Number of inner CV folds
        - n_trials: Number of Optuna trials per inner fold
        - random_state: Random state for reproducibility
        """
        self.X = X
        self.y = y
        self.groups = groups
        self.n_outer_folds = n_outer_folds
        self.n_inner_folds = n_inner_folds
        self.n_trials = n_trials
        self.random_state = random_state
        
        # Encode labels if they're strings
        if y.dtype == 'object':
            self.label_encoder = LabelEncoder()
            self.y = self.label_encoder.fit_transform(y)
        else:
            self.y = y
            self.label_encoder = None
            
        # Initialize CV splitters
        self.outer_cv = GroupKFold(n_splits=n_outer_folds)
        self.inner_cv = GroupKFold(n_splits=n_inner_folds)
        
        # Store results
        self.outer_scores = []
        self.best_configs = []
        
    def define_search_space(self, trial):
        """
        Define the hyperparameter search space for Optuna.
        """
        # Model selection
        model_name = trial.suggest_categorical('model', ['rf', 'xgb'])
        
        # Imbalance handling technique
        imbalance_technique = trial.suggest_categorical(
            'imbalance_technique', ['none','smote', 'smote_tomek', 'tomek_links']
        )
        
        # Model-specific hyperparameters
        if model_name == 'rf':
            params = {
                'model': 'rf',
                'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 20),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
                'max_features': trial.suggest_categorical('max_features', 
                                                        ['sqrt', 'log2', None]),
                'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
                'imbalance_technique': imbalance_technique
            }
        else:  # XGBoost
            params = {
                'model': 'xgb',
                'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
                'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
                'imbalance_technique': imbalance_technique
            }
                
        return params
    
    def apply_imbalance_handling(self, X_train, y_train, technique):
        """
        Apply the specified imbalance handling technique.
        """
        if technique == 'none':
            return X_train, y_train
        elif technique == 'smote':
            sampler = SMOTE(random_state=self.random_state)
            return sampler.fit_resample(X_train, y_train)
        elif technique == 'smote_tomek':
            sampler = SMOTETomek(random_state=self.random_state)
            return sampler.fit_resample(X_train, y_train)
        elif technique == 'tomek_links':
            sampler = TomekLinks()
            return sampler.fit_resample(X_train, y_train)
        else:
            raise ValueError(f"Unknown imbalance technique: {technique}")
    
    def create_model(self, params):
        """
        Create a model based on the parameters.
        """
        if params['model'] == 'rf':
            return RandomForestClassifier(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                min_samples_split=params['min_samples_split'],
                min_samples_leaf=params['min_samples_leaf'],
                max_features=params['max_features'],
                bootstrap=params['bootstrap'],
                random_state=self.random_state,
                n_jobs=-1
            )
        elif params['model'] == 'xgb':
            return XGBClassifier(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                learning_rate=params['learning_rate'],
                subsample=params['subsample'],
                colsample_bytree=params['colsample_bytree'],
                min_child_weight=params['min_child_weight'],
                gamma=params['gamma'],
                reg_alpha=params['reg_alpha'],
                reg_lambda=params['reg_lambda'],
                random_state=self.random_state,
                n_jobs=-1,
                eval_metric='logloss',  # Suppress warnings
                verbosity=0  # Suppress XGBoost output
            )
    
    def calculate_metrics(self, y_true, y_pred, y_pred_proba=None):
        """
        Calculate comprehensive metrics for model evaluation.
        """
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
            'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
            'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
        }
        
        # Add AUC metrics if probabilities are available and it's not multiclass
        if y_pred_proba is not None:
            try:
                if len(np.unique(y_true)) == 2:
                    # Binary classification
                    metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba[:, 1])
                    metrics['pr_auc'] = average_precision_score(y_true, y_pred_proba[:, 1])
                else:
                    # Multiclass classification
                    metrics['roc_auc_ovr'] = roc_auc_score(y_true, y_pred_proba, 
                                                         multi_class='ovr', average='macro')
            except (ValueError, IndexError):
                # Handle cases where AUC cannot be computed
                pass
                
        return metrics
    
    def inner_cv_objective(self, trial, X_train_outer, y_train_outer, groups_train_outer):
        """
        Objective function for inner cross-validation (hyperparameter optimization).
        """
        params = self.define_search_space(trial)
        
        fold_scores = []
        
        for inner_train_idx, inner_val_idx in self.inner_cv.split(
            X_train_outer, y_train_outer, groups_train_outer
        ):
            # Split data
            X_inner_train = X_train_outer[inner_train_idx]
            X_inner_val = X_train_outer[inner_val_idx]
            y_inner_train = y_train_outer[inner_train_idx]
            y_inner_val = y_train_outer[inner_val_idx]
            
            # Scale features (XGBoost doesn't require scaling, but won't hurt)
            scaler = StandardScaler()
            X_inner_train_scaled = scaler.fit_transform(X_inner_train)
            X_inner_val_scaled = scaler.transform(X_inner_val)
            
            # Apply imbalance handling
            X_inner_train_balanced, y_inner_train_balanced = self.apply_imbalance_handling(
                X_inner_train_scaled, y_inner_train, params['imbalance_technique']
            )
            
            # Create and train model
            model = self.create_model(params)
            model.fit(X_inner_train_balanced, y_inner_train_balanced)
            
            # Predict and calculate score
            y_pred = model.predict(X_inner_val_scaled)
            
            # Use F1 score of positive class as the optimization metric (binary classification)
            score = f1_score(y_inner_val, y_pred, pos_label=1, zero_division=0)
            fold_scores.append(score)
        
        return np.mean(fold_scores)
    
    def run_nested_cv(self):
        """
        Run the nested cross-validation procedure.
        """
        print("Starting Nested Cross-Validation...")
        print(f"Outer folds: {self.n_outer_folds}, Inner folds: {self.n_inner_folds}")
        print(f"Total unique groups: {len(np.unique(self.groups))}")
        
        for fold_idx, (train_idx, test_idx) in enumerate(self.outer_cv.split(self.X, self.y, self.groups)):
            print(f"\n=== Outer Fold {fold_idx + 1}/{self.n_outer_folds} ===")
            
            # Split data for outer fold
            X_train_outer = self.X[train_idx]
            X_test_outer = self.X[test_idx]
            y_train_outer = self.y[train_idx]
            y_test_outer = self.y[test_idx]
            groups_train_outer = self.groups[train_idx]
            
            print(f"Training samples: {len(X_train_outer)}, Test samples: {len(X_test_outer)}")
            print(f"Training groups: {len(np.unique(groups_train_outer))}")
            
            # Inner cross-validation for hyperparameter optimization
            study = optuna.create_study(
                direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=self.random_state),
                pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
            )
            
            print(f"Running hyperparameter optimization with {self.n_trials} trials...")
            study.optimize(
                lambda trial: self.inner_cv_objective(trial, X_train_outer, y_train_outer, groups_train_outer),
                n_trials=self.n_trials,
                show_progress_bar=True
            )
            
            best_params = study.best_params
            print(f"Best parameters: {best_params}")
            print(f"Best inner CV score: {study.best_value:.4f}")
            
            # Train final model on entire outer training set with best parameters
            # Note: XGBoost doesn't require scaling, but we apply it for consistency
            scaler = StandardScaler()
            X_train_outer_scaled = scaler.fit_transform(X_train_outer)
            X_test_outer_scaled = scaler.transform(X_test_outer)
            
            # Apply imbalance handling
            X_train_balanced, y_train_balanced = self.apply_imbalance_handling(
                X_train_outer_scaled, y_train_outer, best_params['imbalance_technique']
            )
            
            # Create and train final model
            final_model = self.create_model(best_params)
            final_model.fit(X_train_balanced, y_train_balanced)
            
            # Evaluate on outer test set
            y_pred = final_model.predict(X_test_outer_scaled)
            try:
                y_pred_proba = final_model.predict_proba(X_test_outer_scaled)
            except AttributeError:
                y_pred_proba = None
            
            # Calculate metrics
            fold_metrics = self.calculate_metrics(y_test_outer, y_pred, y_pred_proba)
            
            print(f"Outer fold test metrics:")
            for metric, value in fold_metrics.items():
                print(f"  {metric}: {value:.4f}")
            
            # Store results
            self.outer_scores.append(fold_metrics)
            self.best_configs.append(best_params)
        
        return self._summarize_results()
    
    def _summarize_results(self):
        """
        Summarize the nested CV results.
        """
        print("\n" + "="*50)
        print("NESTED CROSS-VALIDATION RESULTS SUMMARY")
        print("="*50)
        
        # Convert results to DataFrame for easier analysis
        results_df = pd.DataFrame(self.outer_scores)
        
        print("\nOverall Performance (Mean ± Std across outer folds):")
        for metric in results_df.columns:
            mean_score = results_df[metric].mean()
            std_score = results_df[metric].std()
            print(f"  {metric}: {mean_score:.4f} ± {std_score:.4f}")
        
        # Additional reporting for binary classification
        print(f"\nOptimization Metric (F1 of Positive Class):")
        if 'f1_macro' in results_df.columns:
            # Also show how the positive class F1 compares to macro F1
            print(f"  Mean Macro F1: {results_df['f1_macro'].mean():.4f} ± {results_df['f1_macro'].std():.4f}")
            print("  (Note: Optimization used F1 of positive class, not macro F1)")
        
        # Analyze best configurations
        print("\nBest Configurations per Fold:")
        config_summary = defaultdict(list)
        for i, config in enumerate(self.best_configs):
            print(f"  Fold {i+1}: Model={config['model']}, "
                  f"Imbalance={config['imbalance_technique']}")
            config_summary['model'].append(config['model'])
            config_summary['imbalance_technique'].append(config['imbalance_technique'])
        
        print("\nConfiguration Frequency:")
        for key, values in config_summary.items():
            unique_vals, counts = np.unique(values, return_counts=True)
            print(f"  {key}:")
            for val, count in zip(unique_vals, counts):
                print(f"    {val}: {count}/{len(values)} folds")
        
        return {
            'mean_scores': results_df.mean().to_dict(),
            'std_scores': results_df.std().to_dict(),
            'individual_scores': self.outer_scores,
            'best_configs': self.best_configs,
            'summary_df': results_df
        }

# Example usage
if __name__ == "__main__":
    # Example with synthetic data (replace with your actual data)
    from sklearn.datasets import make_classification
    
    # Generate synthetic grouped data
    n_samples = 1000
    n_features = 20
    n_classes = 3
    n_groups = 20
    
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_classes=n_classes,
        n_informative=15,
        n_redundant=5,
        n_clusters_per_class=1,
        class_sep=0.8,
        random_state=42
    )
    
    # Create groups (e.g., subjects/sessions)
    groups = np.repeat(np.arange(n_groups), n_samples // n_groups)
    
    # Initialize and run nested CV
    optimizer = NestedCVOptimizer(
        X=X, 
        y=y, 
        groups=groups,
        n_outer_folds=5,
        n_inner_folds=3,
        n_trials=50,  # Reduced for example
        random_state=42
    )
    
    # Run nested cross-validation
    results = optimizer.run_nested_cv()
    
    print("\nFinal Results Summary:")
    print(f"Mean F1 (macro): {results['mean_scores']['f1_macro']:.4f} ± {results['std_scores']['f1_macro']:.4f}")
    print(f"Mean Accuracy: {results['mean_scores']['accuracy']:.4f} ± {results['std_scores']['accuracy']:.4f}")